# Notebook for reading in baseline period files for NEX-GDDP-CMIP6 models, computing the 75th and 25th percentile temps, saving them out

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#import cartopy.crs as ccrs
from cartopy import crs as ccrs, feature as cfeature
import netCDF4
from netCDF4 import Dataset
from datetime import datetime as dt
import xarray as xr

import warnings
warnings.filterwarnings('ignore')

In [8]:
#import sys
#!conda install --yes --prefix {sys.prefix} dask

## Calculate Baseline Thresholds for Summer & Winter over set of models

In [ ]:
%%time

# comment out as needed
models = ['ACCESS-ESM1-5',
          # 'BCC-CSM2-MR',
          # 'CMCC-ESM2',
          # 'FGOALS-g3',
          # 'GFDL-ESM4',
          # 'GISS-E2-1-G',
          # 'NorESM2-LM'
         ]#,'','','','','','','']


for md in models:

    start = dt.now()
    
    # read in files for 1961-1990
    file_path = '../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/*_historical_*.nc'

    # open all files 1961-2014 for simplicity, select down to 1961-1990 
    #ds = xr.open_mfdataset(file_path, compat='override')
    ds = xr.open_mfdataset(file_path)
    ds_base = ds.sel(time=slice('1961','1990'))

    # load into memory for quantiles
    ds_base.load()

    # 75th percentile for SUMMER
    c_75 = ds_base.tas.quantile(0.75, dim=('time'))
    c_75.attrs["long_name"] = "75th percentile 2m Temperature for the data across available time dimension"
    c_75.attrs["units"] = "K"

    # add in model as dimension
    c_75 = c_75.assign_coords(model=md)
    c_75 = c_75.expand_dims('model')

    # output it
    c_75.to_netcdf('../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/computed_1961-1990-full_75th.nc')

    # 25th percentile for WINTER
    c_25 = ds_base.tas.quantile(0.25, dim=('time'))
    c_25.attrs["long_name"] = "25th percentile 2m Temperature for the data across available time dimension"
    c_25.attrs["units"] = "K"

    # add in model as dimension
    c_25 = c_25.assign_coords(model=md)
    c_25 = c_25.expand_dims('model')

    # output it
    c_25.to_netcdf('../../../Data/CMIP6/NEX-GDDP-CMIP6/'+md+'/regridded/computed_1961-1990-full_25th.nc')    


    # plot them side by side
    fig = plt.figure(figsize=(12,6))

    # SUMMER
    ax1 = fig.add_subplot(1,2,1, projection=ccrs.Robinson(central_longitude=0))
    
    # add coastlines and grid
    ax1.coastlines()
    ax1.gridlines()
    
    # 75th percentile Summer temps
    c_75.plot(
        ax=ax1,
        transform=ccrs.PlateCarree(), # assign map projection
        vmin=230,
        vmax=310, # define temp range for colorbar 230 to 310 K
        cmap="coolwarm"
    )
    ax1.set_title("75th Percentile T2m from 1961 - 1990 for "+md)

    # WINTER
    ax2 = fig.add_subplot(1,2,2, projection=ccrs.Robinson(central_longitude=0))
    
    # add coastlines and grid
    ax2.coastlines()
    ax2.gridlines()
    
    # 25th percentile Winter temps
    c_25.plot(
        ax=ax2,
        transform=ccrs.PlateCarree(), # assign map projection
        vmin=230,
        vmax=310, # define temp range for colorbar 230 to 310 K
        cmap="coolwarm"
    )
    ax2.set_title("25th Percentile T2m from 1961 - 1990 for "+md)

    end = dt.now()
    elapsed = (end-start).total_seconds()/60.0
    print("Time for "+md+" :"+str(elapsed)+" min")